<a href="https://colab.research.google.com/github/EstherMan05/ds2002-fa26/blob/main/notebooks/05-etl-walmart/2026-10-16%20%E2%80%94%20Walmart%20Warm-Up%20SKU%20Mess%20%E2%80%94%20Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Walmart Warm-Up SKU Mess

**Lab — 2026-10-16 · Fall 2026**  

---

## Lab 08 — Walmart Warm-Up (SKU Mess)

This lab uses the **real midterm data** in `../06-midterm-walmart/data/`. It's a warm-up so you hit the ground running on the project. On Kaggle/Colab, upload the data folder or adjust the path.

*If the file isn't found, the cell falls back to a small inline sample so the notebook still runs.*

In [1]:
import pandas as pd
PATH = '../06-midterm-walmart/data/product_catalog.csv'
try:
    cat = pd.read_csv(PATH)
except FileNotFoundError:
    from io import StringIO
    cat = pd.read_csv(StringIO('sku,product_name,category\nPT-12,pop-tarts strawberry,Breakfast Foods\nSKU#459812,Strawberry Pop-Tart,Breakfast\nPOPTART-STRAW,Pop-Tarts Strawberry,Breakfast'))
cat.head()

,sku,product_name,category
0,PT-12,pop-tarts strawberry,Breakfast Foods
1,SKU#459812,Strawberry Pop-Tart,Breakfast
2,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast


### Q1 — How many rows look like Pop-Tart products?

Filter on `product_name` containing `pop`, case-insensitively. Print the count and the matching rows.

In [2]:
is_pop = cat['product_name'].str.lower().str.contains('pop', na=False)
pop = cat[is_pop]

print('pop-tart-like rows:', len(pop))
pop

pop-tart-like rows: 3


,sku,product_name,category
0,PT-12,pop-tarts strawberry,Breakfast Foods
1,SKU#459812,Strawberry Pop-Tart,Breakfast
2,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast


### Q2 — Which SKUs and names all mean *strawberry* Pop-Tart?

List them. Then note which Pop-Tart rows are **not** strawberry — those must stay separate, and saying so is part of the answer.

In [3]:
straw = pop[pop['product_name'].str.lower().str.contains('straw', na=False)]
other_pop = pop[~pop.index.isin(straw.index)]

print('strawberry Pop-Tart variants:')
print(straw[['sku', 'product_name']])
print('\nPop rows that are NOT strawberry:')
print(other_pop[['sku', 'product_name']])

strawberry Pop-Tart variants:
             sku          product_name
0          PT-12  pop-tarts strawberry
1     SKU#459812   Strawberry Pop-Tart
2  POPTART-STRAW  Pop-Tarts Strawberry

Pop rows that are NOT strawberry:
Empty DataFrame
Columns: [sku, product_name]
Index: []


**Pop-Tart rows that are a different product: None in this sample. All three Pop-Tart rows (`PT-12`, `SKU#459812`, `POPTART-STRAW`) are the same strawberry product under different SKUs and spellings, so nothing needs to stay separate. On the full catalog, other flavors such as blueberry would have to stay separate, because a different flavor is a different product, not another code for strawberry.

### Q3 — Build the mapping

A dict from each variant SKU to one canonical SKU. Then apply it in a way that leaves unmapped SKUs as themselves rather than turning them into `NaN` — that single mistake silently deletes most of the catalog.

In [4]:
rows_before = len(cat)

CANON = {sku: 'POPTART-STRAW' for sku in straw['sku']}

cat['canonical_sku'] = cat['sku'].map(CANON).fillna(cat['sku'])   # unmapped SKUs keep themselves
CANON

{'PT-12': 'POPTART-STRAW',
 'SKU#459812': 'POPTART-STRAW',
 'POPTART-STRAW': 'POPTART-STRAW'}

### Q4 — Normalize `category`

Strip and title-case, then show the distinct values before and after with counts. Note any pair that is still two labels for one thing — `Breakfast` and `Breakfast Foods` are not the same string but they are the same category.

In [5]:
print('before:')
print(cat['category'].value_counts(dropna=False))

cat['category'] = cat['category'].str.strip().str.title()
print('\nafter strip + title-case:')
print(cat['category'].value_counts(dropna=False))

# Same thing, different string: a business-level merge, decided explicitly
CAT_MAP = {'Breakfast Foods': 'Breakfast'}
cat['category'] = cat['category'].replace(CAT_MAP)
print('\nafter merging label variants:')
print(cat['category'].value_counts(dropna=False))

before:
category
Breakfast          2
Breakfast Foods    1
Name: count, dtype: int64

after strip + title-case:
category
Breakfast          2
Breakfast Foods    1
Name: count, dtype: int64

after merging label variants:
category
Breakfast    3
Name: count, dtype: int64


### Q5 — Find the fragmentation you have not spotted yet

Hand-scanning a catalog does not scale. **TODO:** build a candidate list programmatically — normalize each product name (lowercase, strip punctuation, sort the words) and group by that, then show any group holding more than one SKU.

This is the workflow from Monday's lecture, and it is how you will actually approach the full catalog on the midterm.

In [6]:
import re

def name_key(s):
    words = re.sub(r'[^a-z0-9\s]', ' ', str(s).lower()).split()
    words = [w[:-1] if w.endswith('s') and len(w) > 3 else w for w in words]   # tarts -> tart
    return ' '.join(sorted(words))

cat['name_key'] = cat['product_name'].map(name_key)

skus_per_key = cat.groupby('name_key')['sku'].nunique()
fragmented = skus_per_key[skus_per_key > 1]

print('names that map to more than one SKU:', len(fragmented))
cat[cat['name_key'].isin(fragmented.index)].sort_values('name_key')[['name_key', 'sku', 'product_name']]

names that map to more than one SKU: 1


,name_key,sku,product_name
0,pop strawberry tart,PT-12,pop-tarts strawberry
1,pop strawberry tart,SKU#459812,Strawberry Pop-Tart
2,pop strawberry tart,POPTART-STRAW,Pop-Tarts Strawberry


### Q6 — Prove the mapping is safe

**TODO:** assert that applying your mapping produced no nulls in the canonical column and did not change the number of rows. Consolidation renames; it never adds or removes.

In [7]:
assert cat['canonical_sku'].notna().all(), 'mapping produced nulls'
assert len(cat) == rows_before, 'mapping changed the row count'
print('mapping is safe:', len(cat), 'rows, no nulls in canonical_sku')

mapping is safe: 3 rows, no nulls in canonical_sku


### Write-up

**a)** Name two product families beyond Pop-Tarts that appear fragmented, and how you would group them.

**b)** Where does consolidation stop being a data-cleaning task and become a business decision? Give one example from this catalog where you genuinely are not sure, and say who you would ask.

A) The real product_catalog.csv wasn't available to me when I ran this, so the notebook used the 3-row fallback sample, which only contains Pop-Tarts and shows no other fragmented families. The one other family I can point to is bottled water, which appeared in the Oct 14 studio data as WATER-24 and BTL-WATER-24 with names like "Bottled Water" and "Bottled Water 24pk". I would group variants like these under one canonical SKU, using the normalized-name key from Q5 to surface candidates and checking each group by hand before merging.

B) Consolidation stops being data cleaning when the question changes from "is this the same string?" to "is this the same product for the business?" Merging "Pop-Tarts Strawberry" and "pop-tarts strawberry" is a spelling fix. Deciding whether a 12-pack and a 24-pack of bottled water count as one product is a merchandising decision: merging them makes units and revenue incomparable, while keeping them apart splits one product's demand across several SKUs. I am not sure which is right, so I would ask the category manager, or whoever owns the product hierarchy, before merging anything.